# Base · Prompted Base · QLoRA Adapter 비교

학습에 사용하지 않은 입력으로 세 조건을 직접 비교합니다. 학습 노트북의 모델을 정리하고 커널을 재시작한 뒤 실행하세요.

## 1. 프로젝트와 task 선택

In [1]:
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(path for path in candidates if (path / "day05").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

TASK = "reply"  # "reply" 또는 "fridge"
print("project:", PROJECT_ROOT, "/ task:", TASK)

project: /home/student/llm-practice/c5-slm / task: reply


## 2. 4-bit base 하나에 두 어댑터 등록

base 모델을 두 번 복제하지 않습니다. reply와 fridge adapter를 이름으로 등록하고 `set_adapter()`로 전환합니다.

In [2]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from day05.life_assistant.config import MODEL_ID, TASKS, generation_prompt, user_text
from day05.life_assistant.inference import generate

assert torch.cuda.is_available()
for name, config in TASKS.items():
    assert config["adapter"].is_dir(), f"{name} adapter 없음: {config['adapter']}"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map={"": 0},
    dtype=torch.bfloat16,
    local_files_only=True,
)
model = PeftModel.from_pretrained(
    base, str(TASKS["reply"]["adapter"]), adapter_name="reply", local_files_only=True
)
model.load_adapter(
    str(TASKS["fridge"]["adapter"]), adapter_name="fridge", local_files_only=True
)
model.eval()
print("registered adapters:", sorted(model.peft_config))
print(f"allocated VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")

/home/student/llm-practice/c5-slm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 434/434 [00:03<00:00, 119.18it/s]


registered adapters: ['fridge', 'reply']
allocated VRAM: 1.46 GiB


## 3. 직접 입력 만들기

In [3]:
if TASK == "reply":
    row = {
        "relation": "직장 동료",
        "situation": "다음 주 월요일까지 부탁받은 자료를 끝내기 어렵다",
        "intent": "목요일까지 연기 요청",
    }
else:
    row = {
        "ingredients": "느타리버섯 한 팩, 달걀 2개, 양파 반 개",
        "condition": "1인분, 15분 이내",
    }

print(user_text(TASK, row))

관계: 직장 동료
상황: 다음 주 월요일까지 부탁받은 자료를 끝내기 어렵다
의도: 목요일까지 연기 요청


## 4. 같은 입력으로 세 조건 생성

- Base: adapter를 끄고 짧은 사용자 입력만 제공
- Prompted base: adapter를 끄고 상세 system 지침 추가
- Adapter: 별도 system 지침 없이 선택한 LoRA 사용

In [4]:
import time

plain_prompt = generation_prompt(TASK, row, with_instruction=False)
instructed_prompt = generation_prompt(TASK, row, with_instruction=True)
model.set_adapter(TASK)

def timed(prompt, adapter_enabled):
    started = time.perf_counter()
    if adapter_enabled:
        answer = generate(model, tokenizer, prompt, TASK)
    else:
        with model.disable_adapter():
            answer = generate(model, tokenizer, prompt, TASK)
    return answer, time.perf_counter() - started

base_answer = timed(plain_prompt, False)
prompted_answer = timed(instructed_prompt, False)
adapter_answer = timed(plain_prompt, True)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [5]:
import pandas as pd

comparison = pd.DataFrame([
    {"condition": "base", "answer": base_answer[0], "seconds": round(base_answer[1], 1)},
    {"condition": "prompted_base", "answer": prompted_answer[0], "seconds": round(prompted_answer[1], 1)},
    {"condition": "adapter", "answer": adapter_answer[0], "seconds": round(adapter_answer[1], 1)},
])
display(comparison)

,condition,answer,seconds
0,base,"김 과장: 이번 주 월요일까지 제출해야 하는 자료가 있는데, 생각보다 양이 많아서 ...",3.6
1,prompted_base,"네, 월요일까지 자료를 완료하기 어려울 것 같습니다. 목요일까지 자료를 완료하는 것...",1.8
2,adapter,다음 주 월요일까지 자료를 제출하기 어려울 것 같습니다. 목요일까지로 연기해 주실 ...,2.2


## 5. 미학습 테스트셋에서 여러 사례 확인

처음에는 `LIMIT=1`로 확인하고, 시간이 허용되면 5로 늘리세요. 사례당 세 번 생성하므로 학습보다 오래 걸릴 수 있습니다.

In [6]:
import json

LIMIT = 1
with TASKS[TASK]["test"].open(encoding="utf-8") as file:
    test_rows = [json.loads(line) for line in file if line.strip()][:LIMIT]

records = []
for index, test_row in enumerate(test_rows, 1):
    plain = generation_prompt(TASK, test_row, with_instruction=False)
    instructed = generation_prompt(TASK, test_row, with_instruction=True)
    model.set_adapter(TASK)
    with model.disable_adapter():
        base_out = generate(model, tokenizer, plain, TASK)
        prompted_out = generate(model, tokenizer, instructed, TASK)
    adapter_out = generate(model, tokenizer, plain, TASK)
    records.append({
        "index": index,
        "input": user_text(TASK, test_row),
        "base": base_out,
        "prompted_base": prompted_out,
        "adapter": adapter_out,
    })

pd.set_option("display.max_colwidth", 120)
display(pd.DataFrame(records))

,index,input,base,prompted_base,adapter
0,1,관계: 아파트 관리실\n상황: 새벽마다 주차장 경보음이 반복되어 잠을 설친다\n의도: 이번 주 안에 원인 점검 요청,"안녕하세요, 관리실입니다. 새벽마다 주차장 경보음이 반복되어 잠을 설친다는 말씀을 들었습니다.","새벽마다 반복되는 주차장 경보음으로 인해 잠을 설친다는 내용을 아파트 관리실에 전달드립니다. 이번 주 안에 원인 점검을 요청드리오니, 빠른 조치 부탁드립니다.",새벽마다 주차장 경보음이 반복되어 잠을 설칩니다. 이번 주 안에 원인을 점검해 주시면 감사하겠습니다.


## 6. 관찰 기록

직접 실행한 뒤 아래 질문에 답해 보세요.

1. Base가 입력의 화자와 수신자를 올바르게 구분했는가?
2. 상세 prompt만으로 목표 형식을 지켰는가?
3. Adapter가 prompt 없이도 형식과 핵심 정보를 보존했는가?
4. 냉장고 모델이 입력에 없는 재료를 몰래 사용하지 않았는가?
5. 학습 데이터 수를 늘린다면 어떤 실패 사례를 추가할 것인가?

## 7. VRAM 정리

In [7]:
import gc

del model, base
gc.collect()
torch.cuda.empty_cache()
print(f"cleanup 후 allocated: {torch.cuda.memory_allocated()/1024**3:.3f} GiB")

cleanup 후 allocated: 0.008 GiB
